In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, lit
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
import requests

In [0]:
spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
def get_hpi_url():
    today = date.today()

    for lag in range(2,4):
        target = today - relativedelta(months=lag)
        url = (
            f"https://publicdata.landregistry.gov.uk/market-trend-data"
            f"/house-price-index-data/UK-HPI-full-file-{target.year}-{target.month:02d}.csv"
        )
        print(f"Trying: {url}")
        response = requests.get(url)
        
        if response.status_code == 200:
            return url, response.content
        print(f"Got {response.status_code} - trying next lag")

    raise RuntimeError("Could not resolve HPI file at lag -2 or -3. Check Land Registry release schedule.")
hpi_url, hpi_bytes = get_hpi_url()

    

In [0]:
import io
import pandas as pd

pdf = pd.read_csv(io.BytesIO(hpi_bytes))
df = spark.createDataFrame(pdf)

print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

In [0]:
def clean_column_name(name):
    name = name.replace("%","Pct")
    if name[0].isdigit():
        name = "_"+name
    return name

df = df.toDF(*[clean_column_name(c) for c in df.columns])
df.printSchema()

In [0]:
from pyspark.sql.functions import lit, current_timestamp

df_bronze = df.withColumn("source_url", lit(hpi_url)) \
            .withColumn("ingest_timestamp", current_timestamp())
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.uk_hpi")
)

print(f"Written {df_bronze.count()} rows to table 'bronze.uk_hpi'")

In [0]:
spark.sql("SELECT RegionName, Date, AveragePrice, SalesVolume FROM bronze.uk_hpi WHERE AreaCode = 'S92000003' ORDER BY Date DESC LIMIT 5").show()

In [0]:
# Bronze ingest complete
# Source: UK HPI full file (Land Registry)
# Resolved URL: {hpi_url}
# Rows written: 149,895
# Table: bronze.uk_hpi
# Columns with problematic names were cleaned before write:
#   % replaced with Pct, digit-prefixed columns prefixed with _
# No filtering, casting, or business logic applied - raw data preserved as-is
# Silver notebook will: filter to Scotland (AreaCode LIKE 'S%'), cast Date,
#   rename problem columns, derive year_month join key

print(f"bronze_01_uk_hpi complete. Source: {hpi_url}")